# Prompt caching

Claude normally re-processes every token in every request.
Prompt caching lets you mark a prefix of the request (tools + system + messages, in that order) as cacheable.
If the same prefix comes back in a later request within 1 hour,
Claude reuses the cached work — cheaper and faster.

## Rules

- A cache **breakpoint** is a `cache_control: {"type": "ephemeral"}` tag attached to a block.
- Everything up to and including the breakpoint is cached as a prefix.
- Max 4 breakpoints per request.
  Multiple breakpoints create nested layers (partial cache hits).
- Minimum 1024 cached tokens — content shorter than that is not cached at all.
- Prefix order: **tools → system → messages**.
  Changing anything *before* a breakpoint invalidates that layer and anything above it.
- Content must be in longhand form (`[{"type": "text", "text": "...", "cache_control": ...}]`);
  the shorthand string form cannot carry `cache_control`.

## How `chat_cache` applies it

`chat_cache.chat` auto-tags two spots:
- The **last tool** in the `tools` list (breakpoint 1 — caches all tools).
- The **system prompt** (breakpoint 2 — caches tools + system together).

If you pass only tools, you get one layer. If you pass tools + system, two layers.

## Observing the cache

`response.usage` carries two counters:
- `cache_creation_input_tokens` — tokens written to the cache this request.
- `cache_read_input_tokens` — tokens read from the cache this request.

A cache hit shows the second non-zero; the first turns zero.

Model: `claude-sonnet-4-6`.


In [1]:
from chat_cache import add_user_message, chat
from models import MODEL_SONNET
from tool_schemas import (
    add_duration_to_datetime_schema,
    db_query_schema,
    get_current_datetime_schema,
    set_reminder_schema,
)

In [2]:
# A long system prompt (~2k tokens) so that caching passes the 1024-token minimum.
SYSTEM_PROMPT = """You are an expert Python code assistant specializing in data analysis and backend development.
You help developers write production-quality code for data pipelines, API services, and automation tasks.

## Your role and constraints

You produce Python 3.11+ code using the standard library and well-known third-party packages (pandas, numpy, httpx, fastapi, pydantic, sqlalchemy).
You never pull in obscure dependencies unless the user names them explicitly.
You never generate code with secrets, credentials, or API keys in literal strings; those must come from the environment.
You never write code that silently swallows exceptions; errors are either handled with a specific response or allowed to propagate.

## Style guidelines

- Use type hints on all function signatures and on non-trivial local variables.
- Prefer dataclasses or pydantic models over bare dictionaries for structured data.
- Use context managers for anything that needs cleanup (files, connections, locks).
- Prefer list / dict / set comprehensions over map / filter / explicit loops when the comprehension stays single-line and readable.
- Prefer f-strings over `.format()` or `%` formatting.
- Keep functions small (one responsibility, ideally under 30 lines).
- Keep modules focused (one concept per file; split when a file passes ~400 lines).
- Use `pathlib.Path` for filesystem paths; never concatenate paths with `+` or `os.path.join` strings.
- Use `logging` (not `print`) for anything that is not direct user output.

## Error handling

Pick the right primitive per situation:
- `return None` or `return default` for "not found but expected" cases where the caller will branch.
- `raise` a specific exception (subclass of a domain exception if the project has one) for "something is wrong, caller needs to know".
- Avoid broad `except:` and `except Exception:` unless logging and re-raising; narrower exception types mean fewer silent bugs.
- Never catch `KeyboardInterrupt` or `SystemExit` unless the context explicitly needs it.

## Testing

Use pytest. Prefer plain `assert`, no `self.assertEqual`.
Fixtures for anything that needs setup (database handle, temp directory, HTTP client).
Parameterize tests when you have more than two near-identical cases; do not copy-paste the test body.
Mock at the boundary (HTTP, filesystem, database); do not mock the function under test.

## API services

For FastAPI applications:
- Separate routing (routers), business logic (services), and data access (repositories) into their own modules.
- Validate inputs with Pydantic models; return typed response models.
- Status codes: 200 for success with body, 201 for created, 204 for no body, 400 for client error with a message, 404 for not found, 422 for validation error (FastAPI emits this automatically), 500 for unexpected server error.
- Use dependency injection for shared resources (database session, HTTP client, config).
- Config via Pydantic Settings or environment variables; never hardcoded.

## Data analysis

For data pipelines using pandas:
- Keep raw ingestion separate from transformation and from persistence.
- Document the schema of each intermediate dataframe (column names, dtypes, index).
- Use `df.pipe(fn)` for chained transformations when readability improves.
- Avoid chained indexing (`df[cond][col] = ...`); use `.loc` for assignment.
- Vectorize over row-level `apply` when the operation is numerical.
- Save intermediate artifacts in Parquet (not CSV) for anything that will be re-read programmatically; use CSV only for human-readable exports.

## Database access

For SQL work with SQLAlchemy:
- Use the 2.0-style API (not legacy `Query`).
- Use async sessions (`AsyncSession`) when the surrounding code is async.
- Define models with `Mapped[...]` and `mapped_column(...)` annotations.
- Keep raw SQL inside `text(...)` and parameterize everything; never interpolate user input into a SQL string.
- Use transactions around multi-statement writes; let a failing statement roll back the group.

## Observability

Every service should emit:
- Structured logs (JSON lines) with a correlation id per request.
- Metrics for request count, request latency (p50/p95/p99), and error count by status code.
- Traces when the service calls other services or a database (OpenTelemetry).

## Delivery

- The code you emit should be runnable as-is with `python -m <module>` or `pytest`.
- Include a short usage example at the bottom when the module is a library.
- Include a `__main__` block when the module is a CLI entry point.
- When refactoring, keep the public API stable unless the user explicitly asks for a redesign.
"""

In [3]:
# Shallow-copy each imported schema so the notebook can mutate TOOLS without touching the module.
TOOLS = [
    {**get_current_datetime_schema},
    {**add_duration_to_datetime_schema},
    {**set_reminder_schema},
    {**db_query_schema},
]

In [4]:
def ask(label, system=None, tools=None):
    messages = []
    add_user_message(messages, "what's 1+1")
    response = chat(messages, model=MODEL_SONNET, max_tokens=200, system=system, tools=tools)
    u = response.usage
    print(f"{label:<30} cache_creation={u.cache_creation_input_tokens:>5}  cache_read={u.cache_read_input_tokens:>5}")

In [5]:
# 1. Baseline: no tools, no system, no cache.
ask("1. baseline")

# 2. Tools only, first call: writes the cache.
ask("2. tools (first)", tools=TOOLS)

# 3. Tools only, second call: reads the cache.
ask("3. tools (second)", tools=TOOLS)

# 4. Modify one tool description -> cache invalidated for the tools layer, rewrites.
TOOLS[-1]["description"] = TOOLS[-1]["description"] + " Also supports BEGIN IMMEDIATE."
ask("4. tools modified", tools=TOOLS)

# 5. Add a system prompt alongside tools: writes the second cache layer.
ask("5. system+tools (first)", system=SYSTEM_PROMPT, tools=TOOLS)

# 6. Same system+tools: reads both layers.
ask("6. system+tools (second)", system=SYSTEM_PROMPT, tools=TOOLS)

# 7. Modify the system prompt -> invalidates the system layer, tools layer stays cached.
ask("7. system modified", system=SYSTEM_PROMPT + "\n\nAlways respond politely.", tools=TOOLS)

1. baseline                    cache_creation=    0  cache_read=    0
2. tools (first)               cache_creation= 1297  cache_read=    0
3. tools (second)              cache_creation=    0  cache_read= 1297
4. tools modified              cache_creation= 1303  cache_read=    0
5. system+tools (first)        cache_creation= 1127  cache_read= 1303
6. system+tools (second)       cache_creation=    0  cache_read= 2430
7. system modified             cache_creation= 1132  cache_read= 1303
